**Transform Constructors Data**
1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId constructor_id)
4. Remove duplicate records
5. Transform values of column nationality to Title Case
6. Write the transformed data to silver constructors table

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql import functions as f

folder_name = 'constructors/constructors_flat'
file_name = 'constructors'
silver_table = f"dev.silver.{file_name}"

# 1. Read Bronze
raw_df = spark.read.format('parquet')\
              .load(f"{bronze_path}/{folder_name}")

# 2. Cast types + Drop url
raw_df = raw_df.withColumn('ingestion_timestamp', f.col('ingestion_timestamp').cast('timestamp'))\
               .drop('url')                       

# 3. Rename columns
raw_df = raw_df.withColumnRenamed('constructorId','constructor_id')\
               .withColumnRenamed('circuitId', 'circuit_id')

# 4. Filter null business key
raw_df = raw_df.filter(f.col('constructor_id').isNotNull())

# 5. Remove duplicates
raw_df = raw_df.dropDuplicates(['constructor_id'])     

# 6. Title Case
raw_df = raw_df.withColumn('nationality', f.initcap(f.col('nationality')))
# 7. Write to Silver
raw_df.write\
      .format('delta')\
      .mode('overwrite')\
      .saveAsTable(silver_table)